In [5]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import os
import json

In [10]:
df = pd.read_csv("csv's/mlb_2024-2026.csv")

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nDate range: {df['game_date'].min()} → {df['game_date'].max()}")
print(f"\nPitch type distribution:\n{df['pitch_type'].value_counts()}")

Shape: (1784664, 117)

Columns:
['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'babip_value', 'iso_value', 'l

In [9]:
STAT_MAP = {
    'release_spin_rate' : 'spin_rate',
    'pfx_z'             : 'induced_vertical_break',
    'pfx_x'             : 'horizontal_break',
    'release_extension' : 'extension',
    'release_pos_z'     : 'release_height',
    'release_pos_x'     : 'release_side',
    'release_spin_axis' : 'spin_axis',
}

FEATURES = [
    'release_speed',
    'spin_rate',
    'induced_vertical_break',
    'horizontal_break',
    'extension',
    'release_height',
    'release_side',
    'spin_axis',
]

FLIP_COLS = ['horizontal_break', 'release_side']

df = df.rename(columns=STAT_MAP)

# Normalize LHP → RHP convention
df.loc[df['p_throws'] == 'L', FLIP_COLS] *= -1
df.loc[df['p_throws'] == 'L', 'spin_axis'] = (360 - df.loc[df['p_throws'] == 'L', 'spin_axis']) % 360

# Drop rows missing the target variable
df = df.dropna(subset=['delta_run_exp']).copy()

print(f"Rows after cleaning: {len(df):,}")
print(f"\nNull counts:\n{df[FEATURES + ['delta_run_exp']].isnull().sum()}")
print(f"\nPitch type counts after cleaning:\n{df['pitch_type'].value_counts()}")

Rows after cleaning: 1,778,692

Null counts:
release_speed             30452
spin_rate                 36858
induced_vertical_break    30453
horizontal_break          30586
extension                 32328
release_height            30451
release_side              30451
spin_axis                 36860
delta_run_exp                 0
dtype: int64

Pitch type counts after cleaning:
pitch_type
FF    553840
SI    276048
SL    254143
CH    181021
FC    137654
ST    130248
CU    112686
FS     56382
KC     30546
SV      8421
EP      1813
FA      1613
KN      1378
FO      1291
CS       520
SC       197
PO       125
UN        18
Name: count, dtype: int64
